In [7]:
### PACKAGES ###

from pathlib import Path
import random 
import shutil
import cv2 as cv
from PIL import Image
import os
from tqdm import tqdm 


In [8]:
### ADD SQUARED PADDING TO ROI ###

## INPUTS ##
INPUT_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Training data/Monitoring_training_data/OG_training_data_2"
DEST_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Training data/Monitoring_training_data/OG_training_data_2_squered"

os.makedirs(DEST_FOLDER, exist_ok=True)

## FUNCTIONS ##

def collect_image_files(input_dir):
    """
    Collect all image files recursively from input directory.
    
    Arguments
    - input_dir: Path to the input directory containing images.

    Returns:
    - list of files with extentions: ".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"
    """
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(input_dir)
        for file in files
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"))
    ]


def check_files(file_list):
    """
    Quality check on images

    Arguments:
    - file_list: list of image files

    Returns
    - valid files
    """
    valid_files = []
    for filename in tqdm(file_list, desc="Quality checking images"):
        try:
            with Image.open(filename) as img:
                img.verify()
            valid_files.append(filename)
        except Exception:
            continue
    return valid_files

def add_squered_padding_to_roi(image_path):
    
    # Load segmented image
    segmented_image = cv.imread(image_path)

    # Get original dimensions
    h_seg, w_seg = segmented_image.shape[:2]

    # Determine the target size (largest dimension)
    max_side = max(h_seg, w_seg)

    # Compute padding amounts
    top = (max_side - h_seg) // 2
    bottom = max_side - h_seg - top
    left = (max_side - w_seg) // 2
    right = max_side - w_seg - left
    # Add the border 
    padded = cv.copyMakeBorder(
        segmented_image,
        top, bottom, left, right,
        borderType=cv.BORDER_CONSTANT,
        value=(0, 0, 0)  # Black border
    )
    return padded



In [ ]:
### Padding ###

image_files = collect_image_files(INPUT_FOLDER)

# mac often create phantom files, so we need to check the files before processing
valid_images = check_files(image_files)

os.makedirs(DEST_FOLDER, exist_ok=True)

for image in tqdm(valid_images, desc="Padding images", total=len(valid_images)):
    
    # Extract taxa name and file name
    base_name = os.path.splitext(os.path.basename(image))[0]
    taxa_name = os.path.basename(os.path.dirname(image))
    
    taxa_dir = os.path.join(DEST_FOLDER, taxa_name)
    os.makedirs(taxa_dir, exist_ok=True)
    
    # Add padding
    recent_img = add_squered_padding_to_roi(image)
    
    # Saving padded images 
    save_name = f"sq_{base_name}.png"
    save_path = os.path.join(taxa_dir, save_name)
    
    cv.imwrite(save_path, recent_img)

Padding images: 100%|██████████| 2714/2714 [11:55<00:00,  3.79it/s]


In [ ]:
### recognize phantom files

copied_images = collect_image_files(INPUT_FOLDER)
valid__copies = check_files(copied_images)
invalid_images = set(copied_images) - set(valid__copies)

print(f"Valid images: {len(valid__copies)}")
print(f"Invalid images: {len(invalid_images)}")

In [ ]:
### remove phantom files

removed_count = 0
for img in tqdm(invalid_images, desc="Removing invalid files", total=len(invalid_images)):
    os.remove(img)
    removed_count += 1
    
print(f"Removed {removed_count} invalid files.")

In [14]:
### Split into train/val/test sets ###

## INPUTS ##
INPUT_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Training data/Monitoring_training_data/OG_training_data_2_squered_pooled"
OUTPUT_FOLDER = "/Volumes/research/LU24A1037-Jellyscope/Jellyscope/Training data/Monitoring_training_data/OG_training_data_2_squered_pooled_split"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

splits = {"train": 0.8, "val": 0.2, "test": 0.0}

## MAIN SCRIPT ##
random.seed(42)  # reproducible splits

for class_dir in Path(INPUT_FOLDER).iterdir():
    if not class_dir.is_dir():
        continue
    
    images = list(class_dir.glob("*.*"))
    
    n_total = len(images)
    n_train = round(splits["train"] * n_total)
    n_val = round(splits["val"] * n_total)
    n_test = n_total - n_train - n_val

    random.shuffle(images)
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train + n_val]
    test_imgs = images[n_train + n_val:]

    for split_name, split_imgs in zip(["train", "val", "test"], [train_imgs, val_imgs, test_imgs]):
        split_dir = Path(OUTPUT_FOLDER) / split_name / class_dir.name
        split_dir.mkdir(parents=True, exist_ok=True)
        
        for img in split_imgs:
            shutil.copy2(img, split_dir / img.name)

In [17]:
### recognize phantom files

copied_images = collect_image_files(OUTPUT_FOLDER)
valid__copies = check_files(copied_images)
invalid_images = set(copied_images) - set(valid__copies)

print(f"Valid images: {len(valid__copies)}")
print(f"Invalid images: {len(invalid_images)}")

Quality checking images:   2%|▏         | 43/2714 [00:02<02:56, 15.16it/s]


KeyboardInterrupt: 

In [16]:
### remove phantom files

removed_count = 0
for img in tqdm(invalid_images, desc="Removing invalid files", total=len(invalid_images)):
    os.remove(img)
    removed_count += 1
    
print(f"Removed {removed_count} invalid files.")

Removing invalid files: 100%|██████████| 2714/2714 [00:38<00:00, 70.03it/s]

Removed 2714 invalid files.


In [ ]:
# Augment data by flipping images horizontally and vertically, and saving them with new names

#### FUNCTIONS ####
# to augment images #

def rotate_image(image, angle=(90, 270)):
    """
    Rotate image either to 90 or0 270 degrees with a given probability.

    Arguments:
    - image: input image as a numpy array
    - angle: either 90 or 270 degrees

    Returns:
    - rotated image
    """

    chosen_angle = np.random.choice(angle)
    if chosen_angle == 90:
        return cv.rotate(image, cv.ROTATE_90_CLOCKWISE)
    else:
        return cv.rotate(image, cv.ROTATE_90_COUNTERCLOCKWISE)
    

def local_blur(image, max_rect_ratio=0.3, max_ksize=51):
    """
    Apply Gaussian blur to a random rectangle in the image.
    
    Args:
        image: np.array (H x W x C)
        max_rect_ratio: max fraction of image to cover in rectangle
        max_ksize: max kernel size for Gaussian blur (odd number)
    
    Returns:
        augmented image
    """
    h, w = image.shape[:2]

    # Random rectangle size
    rw = int(np.random.uniform(0.1, max_rect_ratio) * w)
    rh = int(np.random.uniform(0.1, max_rect_ratio) * h)

    # Random top-left corner
    x1 = np.random.randint(0, w - rw)
    y1 = np.random.randint(0, h - rh)
    x2, y2 = x1 + rw, y1 + rh

    # Random odd kernel size for blur
    ksize = np.random.choice(range(3, max_ksize, 2))

    # Copy image and apply blur locally
    blurred_image = image.copy()
    blurred_image[y1:y2, x1:x2] = cv.GaussianBlur(image[y1:y2, x1:x2], (ksize, ksize), 0)

    return blurred_image

# not used in the end, but can be useful for future data augmentation
def cutmix(image1, image2, alpha=0.3):
    """
    Mix a random patch from image2 into image1.
    
    Args:
        image1: np.array, target image
        image2: np.array, source image (same shape as image1)
        alpha: max fraction of image to cut
    
    Returns:
        mixed image
    """
    
    h1, w1 = image1.shape[:2]
    h2, w2 = image2.shape[:2]

    # Patch size: no larger than min(image1, image2) * alpha
    rw = max(1, int(np.random.uniform(0.1, alpha) * min(w1, w2)))
    rh = max(1, int(np.random.uniform(0.1, alpha) * min(h1, h2)))

    # Top-left corner in image1
    x1 = np.random.randint(0, w1 - rw + 1)
    y1 = np.random.randint(0, h1 - rh + 1)

    # Top-left corner in image2
    xs1 = np.random.randint(0, w2 - rw + 1)
    ys1 = np.random.randint(0, h2 - rh + 1)

    # Compute bottom-right corners
    x2, y2 = x1 + rw, y1 + rh
    xs2, ys2 = xs1 + rw, ys1 + rh

    mixed = image1.copy()
    mixed[y1:y2, x1:x2] = image2[ys1:ys2, xs1:xs2]

    return mixed


In [ ]:
# collect imagages (fill path) from split data folder
input_dir = r"R:\LU24A1037-Jellyscope\Jellyscope\Training data\Monitoring_training_data\augment_training_data"

image_files = [
        os.path.join(root, file)
        for root, _, files in os.walk(input_dir)
        for file in files
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"))
    ]

print(f"Found {len(image_files)} images to augment.")

In [ ]:
#### Augment images ####
# will double the dataset by applying either rotation or flip, and then either local blur or cutmix to make variaitons more distinct

### chance to roteate or flip images
chance_to_rotate_or_flip = 0.5
chance_to_blur = 1

for file in tqdm(image_files, desc="Augmenting images"):
    
    # directory and filename
    dir_name = os.path.dirname(file)
    # filename without extension
    base_name = os.path.splitext(os.path.basename(file))[0]

    
    # Read image
    image = cv.imread(file)

    # Apply augmentations: either rotate or flip
    if np.random.rand() < chance_to_rotate_or_flip:
        augmented_image = rotate_image(image, angle=(90, 270))
    
    else:
        augmented_image = cv.flip(image, 1)  # Horizontal flip

    # Apply either local blur or cutmix to make variaitons more distinct
    if np.random.rand() < chance_to_blur:
        augmented_image = local_blur(augmented_image, max_rect_ratio=0.3, max_ksize=31) 

    # Save augmented image
    augmented_image_path = os.path.join(dir_name, f"{base_name}_aug.png")
    cv.imwrite(augmented_image_path, augmented_image)